### Architectural Perspectives on LLM Embedding Matrices: GPT-2 vs. BERT

This notebook provides a complete hands-on exploration of the token embedding matrices inside modern transformer-based language models (GPT-2 and BERT).

**Key Topics Covered:**
1. Full Model Loading: Importing complete PyTorch transformer architectures via Hugging Face `transformers`.
2. Vocabulary Analysis: Comparing Byte-Pair Encoding (BPE) vs. WordPiece token lengths.
3. Embedding Extraction: Extracting raw weight matrices from model graph objects using `.detach()`.
4. Vector Orthogonality: Testing linear correlation between identical tokens across models.
5. Distributional Geometry: Calculating and visualizing token-wise means and variances.
6. Contextual Deformation: Measuring how initial static embeddings shift across self-attention layers.

***Prerequisites***
Install the required packages before running:
```bash
pip install torch transformers numpy matplotlib seaborn

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, GPT2Model, BertModel

# Set seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

Explore the Model Architectures

In [ ]:
print("Loading GPT-2 (small) and BERT (base uncased)...")
tokenizer_gpt = AutoTokenizer.from_pretrained("gpt2")
model_gpt = GPT2Model.from_pretrained("gpt2")

tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased")
model_bert = BertModel.from_pretrained("bert-base-uncased")

print(model_gpt)
print(model_bert)

Vocabulary & Token Length Analysis

In [ ]:
vocab_size_gpt = tokenizer_gpt.vocab_size
vocab_size_bert = tokenizer_bert.vocab_size

print(f"GPT-2 Vocabulary Size: {vocab_size_gpt:,} tokens")
print(f"BERT Vocabulary Size:  {vocab_size_bert:,} tokens")

# Compute string character lengths for all vocabulary items
lengths_gpt = [len(tokenizer_gpt.decode([i])) for i in range(vocab_size_gpt)]
lengths_bert = [len(tokenizer_bert.decode([i])) for i in range(vocab_size_bert)]

# Plot Token Length Distributions
plt.figure(figsize=(10, 4))
plt.hist(lengths_gpt, bins=range(1, 25), alpha=0.5, label='GPT-2 (BPE)', color='tab:orange', density=True)
plt.hist(lengths_bert, bins=range(1, 25), alpha=0.5, label='BERT (WordPiece)', color='tab:blue', density=True)
plt.title("Token Character Length Distributions")
plt.xlabel("Character Length per Token")
plt.ylabel("Density")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

Extracting and Visualizing Embedding Weight Matrices

To perform numeric array manipulations, weight tensors must be explicitly detached from PyTorch's automatic differentiation computational graph using .detach().

In [ ]:
# Detach PyTorch tensors from graph and convert to NumPy arrays
W_gpt = model_gpt.wte.weight.detach().numpy() # Shape: (50257, 768)
W_bert = model_bert.embeddings.word_embeddings.weight.detach().numpy() # Shape: (30000, 768)

print(f"W_gpt (GPT-2 Embedding Matrix) Shape: {W_gpt.shape}")
print(f"W_bert (BERT Embedding Matrix) Shape:  {W_bert.shape}")

# Visualizing heatmaps with identical color scaling (-0.15 to +0.15)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(W_gpt[:500, :].T, aspect='auto', cmap='viridis', vmin=-0.15, vmax=0.15)
axes[0].set_title("GPT-2 Embeddings (First 500 Tokens x 768 D)")
axes[0].set_xlabel("Token Index")
axes[0].set_ylabel("Embedding Dimension")

im1 = axes[1].imshow(W_bert[:500, :].T, aspect='auto', cmap='viridis', vmin=-0.15, vmax=0.15)
axes[1].set_title("BERT Embeddings (First 500 Tokens x 768 D)")
axes[1].set_xlabel("Token Index")
axes[1].set_ylabel("Embedding Dimension")

fig.colorbar(im1, ax=axes.ravel().tolist(), orientation='vertical', shrink=0.8, label="Weight Value")
plt.show()

Cross-Model Token Correlation Analysis for a given word.

In [ ]:
target_word = "the"

# Encode the target word using each tokenizer
id_gpt = tokenizer_gpt.encode(target_word)[0]
id_bert = tokenizer_bert.encode(target_word)[0]

print(f"Token ID for '{target_word}' in GPT-2: {id_gpt}")
print(f"Token ID for '{target_word}' in BERT:  {id_bert}")

# Extract respective 768-dimensional vectors
vec_gpt = W_gpt[id_gpt]
vec_bert = W_bert[id_bert]

# Calculate Pearson correlation coefficient
corr_coef = np.corrcoef(vec_gpt, vec_bert)[0, 1]
print(f"Pearson Correlation Coefficient for '{target_word}': {corr_coef:.4f}")

# Scatter plot across all 768 dimensions
plt.figure(figsize=(6, 6))
plt.scatter(vec_gpt, vec_bert, alpha=0.6, color='purple', edgecolors='none', s=25)
plt.title(f"Dimension-wise Comparison for Token: '{target_word}'\nCorrelation = {corr_coef:.4f}")
plt.xlabel("GPT-2 Vector Dimensions (d=768)")
plt.ylabel("BERT Vector Dimensions (d=768)")
plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

Statistical Geometry: Distribution of Vector Means & Variances

In [ ]:
# Compute row-wise means and variances across embedding dimensions (axis=1)
means_gpt = np.mean(W_gpt, axis=1)
vars_gpt = np.var(W_gpt, axis=1)

means_bert = np.mean(W_bert, axis=1)
vars_bert = np.var(W_bert, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Distribution of Means
axes[0].hist(means_gpt, bins=50, alpha=0.6, label='GPT-2', color='tab:orange', density=True)
axes[0].hist(means_bert, bins=50, alpha=0.6, label='BERT', color='tab:blue', density=True)
axes[0].set_title("Distribution of Token Vector Means")
axes[0].set_xlabel("Mean Value across 768 Dimensions")
axes[0].set_ylabel("Density")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# Distribution of Variances
axes[1].hist(vars_gpt, bins=50, alpha=0.6, label='GPT-2', color='tab:orange', density=True)
axes[1].hist(vars_bert, bins=50, alpha=0.6, label='BERT', color='tab:blue', density=True)
axes[1].set_title("Distribution of Token Vector Variances")
axes[1].set_xlabel("Variance across 768 Dimensions")
axes[1].set_ylabel("Density")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

Dynamic Contextual Transformations Across Layers

Note that GPT-2 replaces space characters (' ') with the unicode character Ġ (U+0120, Latin Capital Letter G with Dot Above).

In [ ]:
sentence = "The bank of the river was muddy."
tokens = tokenizer_gpt.tokenize(sentence)
input_ids = tokenizer_gpt.encode(sentence, return_tensors="pt")

# Pass forward with hidden state recording
with torch.no_grad():
    outputs = model_gpt(input_ids, output_hidden_states=True)
    hidden_states = outputs.hidden_states  # Tuple of 13 tensors

print(f"Captured {len(hidden_states)} hidden state layers (Embedding + 12 Blocks).")

# Target token: 'bank' at index 1
target_token_idx = 1
initial_emb = hidden_states[0][0, target_token_idx, :]
final_layer_emb = hidden_states[-1][0, target_token_idx, :]

cos_sim = torch.nn.functional.cosine_similarity(initial_emb, final_layer_emb, dim=0).item()
print(f"Target Token: '{tokens[target_token_idx]}'")
print(f"Cosine Similarity (Initial Static Vector vs. Final Contextual Vector): {cos_sim:.4f}")

# Layer-by-layer similarity decay curve
layer_sims = []
for layer_idx in range(len(hidden_states)):
    layer_vec = hidden_states[layer_idx][0, target_token_idx, :]
    sim = torch.nn.functional.cosine_similarity(initial_emb, layer_vec, dim=0).item()
    layer_sims.append(sim)

plt.figure(figsize=(8, 4))
plt.plot(range(13), layer_sims, marker='o', color='tab:red', linewidth=2)
plt.title(f"Layer-wise Cosine Similarity Decay for Token '{tokens[target_token_idx]}'")
plt.xlabel("Transformer Layer Depth (0 = Embedding, 12 = Final Block)")
plt.ylabel("Cosine Similarity with Input Embedding")
plt.grid(True, linestyle="--", alpha=0.5)
plt.xticks(range(13))
plt.tight_layout()
plt.show()